In [29]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

In [30]:
df = pd.read_csv("../data/raw/job_descriptions.csv")

print("Shape:", df.shape)
df.head()

Shape: (1615940, 23)


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,Job Posting Date,Preference,Contact Person,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,2022-04-24,Female,Brandon Cunningham,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,2022-12-19,Female,Francisco Larsen,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,2022-09-14,Male,Gary Gibson,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,2023-02-25,Female,Joy Lucero,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,2022-10-11,Female,Julie Johnson,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


In [31]:
print(df.columns.tolist())

['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', 'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']


In [32]:
cols_to_keep = [
    "Job Title",
    "Job Description",
    "Experience", 
    "Qualifications",
    "Responsibilities",
    "Preference",
]

df = df[cols_to_keep].copy()

In [33]:
df.columns = [
    "title",
    "description",
    "experience",
    "qualifications",
    "responsibilities",
    "preference"
]
# before dropping
initial_rows = df.shape[0]

# drop rows with ANY missing values
df_clean = df.dropna()

# after dropping
final_rows = df_clean.shape[0]

# number of rows dropped
dropped_rows = initial_rows - final_rows

print("Initial rows:", initial_rows)
print("Final rows:", final_rows)
print("Rows dropped:", dropped_rows)

Initial rows: 1615940
Final rows: 1615940
Rows dropped: 0


In [34]:
df["text"] = (
    df["description"].fillna("").astype(str) + " " +
    df["experience"].fillna("").astype(str) + " " +
    df["qualifications"].fillna("").astype(str) + " " +
    df["responsibilities"].fillna("").astype(str)
)

def clean_text(text):
    text = str(text or "").lower()
    text = re.sub(r"http\S+", " ", text)              # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)         # remove special chars
    text = re.sub(r"\s+", " ", text).strip()         # normalize spaces
    return text

df["clean_text"] = df["text"].apply(clean_text)

def clean_for_embeddings(text):
    text = str(text or "").lower()

    # remove explicit label leakage
    text = re.sub(r"\b(male|female|both)\b", " ", text)

    # remove structured metadata artifacts
    text = re.sub(r"\bto years\b", " ", text)
    text = re.sub(r"\byears\b", " ", text)

    # remove repeated degree abbreviations
    text = re.sub(
        r"\b(b com|m com|bba|ba|bs|ms|msc|mba|m tech|b tech|phd|doctorate)\b",
        " ",
        text,
    )

    # keep useful tech tokens a bit better than before
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\.\+#_\-\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["embedding_text"] = (
    df["description"].fillna("").astype(str) + " " +
    df["responsibilities"].fillna("").astype(str)
).apply(clean_for_embeddings)

print(df[["clean_text", "embedding_text"]].head())
print("Avg clean_text length:", df["clean_text"].str.split().apply(len).mean())
print("Avg embedding_text length:", df["embedding_text"].str.split().apply(len).mean())

                          title  \
0  Digital Marketing Specialist   
1                 Web Developer   
2            Operations Manager   
3              Network Engineer   
4                 Event Manager   

                                         description     experience  \
0  Social Media Managers oversee an organizations...  5 to 15 Years   
1  Frontend Web Developers design and implement u...  2 to 12 Years   
2  Quality Control Managers establish and enforce...  0 to 12 Years   
3  Wireless Network Engineers design, implement, ...  4 to 11 Years   
4  A Conference Manager coordinates and manages c...  1 to 12 Years   

  qualifications                                   responsibilities  \
0         M.Tech  Manage and grow social media accounts, create ...   
1            BCA  Design and code user interfaces for websites, ...   
2            PhD  Establish and enforce quality control standard...   
3            PhD  Design, configure, and optimize wireless netwo...   
4      

In [35]:
df.to_csv("../data/job_postings_clean.csv", index=False)